# Configurable waveforms — one configuration, Run All

Select **SCENARIO**, **MODE**, then **EVIDENCE_MODE**. Every DD figure uses one
selected **Learned** W per N/run, alongside OTFS and AFDM. No per-SNR switching.

- **load_existing** loads a verified archived basis or your new DD checkpoint.
- **train_new** actually trains DD on MPS/FP32, selects once on validation, then
  evaluates independently. This bounded multipath concentration surrogate is a new recipe;
  it does not reproduce historical training or guarantee better BER.
- Residual CFO training is **UNAVAILABLE**: only its verified rectangular W/R
  and fresh direct-analysis evaluation are supported. No DD trainer substitution.

**Frozen-basis evaluation knobs** change H/test SNR/sampling, not W. Changing W,
channel law, K/support or grid cannot relabel old BER: use **fresh**. **saved_replay**
is exact nominal DD archive replay using its original CPU double reference path.
All fresh simulations/training use MPS FP32 real/imaginary pairs, fallback disabled.

Restart kernel/clear outputs after config changes. Tracked notebook is output-free.

In [ ]:
# CONFIGURATION — all user controls here. Restart kernel, then Run All.
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "advisor").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT))
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ["MPLCONFIGDIR"] = str(ROOT / "local/.matplotlib")

SCENARIO = "dd_v1"                   # "dd_v1" or "residual_cfo"; physically distinct.
MODE = "load_existing"               # "load_existing" or DD "train_new".
EVIDENCE_MODE = "saved_replay"        # "fresh" required for train_new/CFO/modified scenarios.
N = 64                              # DD: 64/128/256, own W per N. CFO: 64 only.
K_FRACTION = 3 / 8                   # DD row support fraction; K must be integral.
K = round(N * K_FRACTION)
if N * K_FRACTION != K:
    raise ValueError("N*K_FRACTION must be integral")
SUPPORT_POLICY = "frozen"            # "frozen" or explicit load_existing "recalibrate".
# Frozen: archived support, or new W's train-only support. K changes require
# train_new/recalibrate; no silent mask changes. Full receiver always keeps all A.
MODEL = dict(n=N, paths=3, delay_range=(0.0, 4.0), doppler_range=(-0.75, 0.75))
# DD-v1 only: finite circular fractional delay then negative Doppler; normalized H.
# Allowed paths 1–6; delay [0,8] samples, Doppler [-2,2] DFT bins, increasing ranges.

SCALING_ARTIFACT_DIR = ROOT / "local/scaling-artifacts"
SCALING_BUNDLE = ROOT / "local/advisor-scaling-artifacts.tar.gz"
ARTIFACT_DIR = ROOT / "local/artifacts"             # CFO legacy bundle only.
ARTIFACT_BUNDLE = ROOT / "local/advisor-n64-artifacts.tar.gz"
CHECKPOINT = None                    # DD .npz under local/: load, or exclusive train save.
# None: archived Learned when loading; auto-save selected checkpoint in train archive.
OUTPUT_ROOT = ROOT / "local/notebook_runs"
SAVE_RUN = True                      # Ignored local archives; figures always inline.

# Display only: native IDs, no basis/receiver/BER changes.
BASIS_COLUMNS = {64:(0,11,15,42,44,52),128:(0,8,22,46),256:(0,40,87,148)}
EXAMPLE_COLUMNS = {64:(0,15,44),128:(0,8,22,46),256:(0,40,87,148)}
DISPLAY = dict(basis_columns=BASIS_COLUMNS.get(N,(0,)),
               example_columns=EXAMPLE_COLUMNS.get(N,(0,)), snr_max_db=25.0)
if MODE == "train_new" or CHECKPOINT is not None or SUPPORT_POLICY == "recalibrate":
    DISPLAY = dict(basis_columns=(0,1), example_columns=(0,), snr_max_db=25.0)
# At most two individual examples per discovered group; groups are derived, not forced.
OVERLAY8_COLUMNS = tuple(range(0,N,max(1,N//8)))
HALF_COLUMNS = tuple(range(0,N,2))
PLOT_VARIANTS = ("overlay8_complex","overlay_half_complex",
                 "overlay8_energy_magnitude","overlay_half_energy_magnitude","grouped_spectra")
REVIEW = dict(variants=PLOT_VARIANTS,overlay8_columns=OVERLAY8_COLUMNS,half_columns=HALF_COLUMNS)

# Test only: changing these never retrains/selects W. Es/N0, raw QPSK/Gray16QAM.
EVALUATION = None
if EVIDENCE_MODE == "fresh":
    EVALUATION = dict(seed=303, matrix_channels=2, matrix_index=0,
        max_channels=8, min_channels=4, batch_channels=2, blocks_per_channel=2,
        snrs_db=(0.0,10.0,18.0,25.0), modulations=("QPSK","16QAM"),
        target_errors=100, max_seconds_per_point=30.0,
        diagnostic_atoms=8, diagnostic_channels=8)
# Every waveform/receiver shares H/bits/physical noise and the same stopped prefix.
# Time checked after each bounded batch; one batch can exceed cap. Small runs are
# uncertain, including zero-error points; tables keep counts/bounds/stopping flags.

# Training knobs: used ONLY by train_new. Separate train/validation laws permitted.
# Objective = uncaptured mean normalized multipath coupling energy under own train-only support.
# Validation chooses one step (including zero); test BER cannot choose/retrain W.
TRAINING = None
if MODE == "train_new":
    TRAINING = dict(model=dict(MODEL), validation_model=dict(MODEL),
        seed=101, validation_seed=202, iterations=20, learning_rate=0.02,
        layers=4, initialization="OFDM", batch_channels=8, validation_channels=8,
        validate_every=5, max_seconds=120.0, group_threshold=0.15)
# New recipe: alternating time/Fourier phase layers, unitary by construction.
# Seeds train/validation/test must differ; namespaces also separate matrix/BER/diagnostics.
DISCOVERY = None
if SUPPORT_POLICY == "recalibrate":
    DISCOVERY = dict(model=dict(MODEL), seed=404, channels=16, group_threshold=0.15)

# Residual CFO uses 64 physical samples / 45 streams; no DD N/K or W^H assumption.
if SCENARIO == "residual_cfo":
    MODEL = dict(n=N, streams=45, cfo_range=(-0.1,0.1))  # positive phase; range within [-.5,.5].
    K, SUPPORT_POLICY, DISCOVERY, REVIEW = None, None, None, None
    DISPLAY = dict(basis_columns=(0,22,44), snr_max_db=25.0)
    if EVALUATION is not None:
        EVALUATION = {key:value for key,value in EVALUATION.items()
                      if key not in ("diagnostic_atoms","diagnostic_channels")}

## 1. Validate and select one Learned

Exact replay requires the sealed size-specific W/support and nominal DD law.
Fresh evaluation loads W without changing it; explicit support recalibration uses
an independent discovery bank and is labeled. Training creates its own W/support.
N64/128/256 at K/N=3/8 use K24/48/96; matrices are never resized.

Selected DD checkpoint = lowest held-out multipath leakage using train-only support,
earliest tie, including initialization. Training is not BER optimization. Traces
and selection metadata stay in the local checkpoint/archive. No per-SNR selection.

CFO instead loads verified W[64,45], R[45,64]. Unsupported CFO training, DD sparse
receivers, checkpoint substitution and archived BER replay fail explicitly.

In [ ]:
from IPython.display import display, JSON
import pandas as pd
from advisor.entry import start_workflow

workflow = None  # Failed selection must never reuse prior run state.
workflow = start_workflow(scenario=SCENARIO,mode=MODE,evidence_mode=EVIDENCE_MODE,
    model=MODEL,display=DISPLAY,evaluation=EVALUATION,training=TRAINING,
    k=K,support_policy=SUPPORT_POLICY,discovery=DISCOVERY,checkpoint=CHECKPOINT,
    workspace=ROOT,output_root=OUTPUT_ROOT,save_run=SAVE_RUN,
    scaling_artifacts=SCALING_ARTIFACT_DIR,scaling_bundle=SCALING_BUNDLE,
    artifacts=ARTIFACT_DIR,bundle=ARTIFACT_BUNDLE,review=REVIEW)
display(JSON(workflow.summary(),expanded=True))

## 2. Physical H and complex effective A

DD-v1: Hraw=ΣhₚΔ(νₚ)Π(τₚ), negative signs, delay first;
H=Hraw/√(‖Hraw‖F²/N), A=WᴴHW. Same normalized H for every waveform.
CFO: H=diag(exp(+j2πεn/64)), no DD normalization, A=RHW, distinct R.
Physical white noise after R has covariance σ²RRᴴ, not σ²I.

Matrices remain complex exports in memory; heatmaps show entry power. Matrix,
BER and diagnostic test banks have separate RNG namespaces.

In [ ]:
channels, A = workflow.generate_channels()
H, Hraw, W = channels.h, channels.raw, workflow.ws
display(workflow.matrix_summary())
display(workflow.path_table())
workflow.show("channel_matrices_native")

## 3. Basis spectra and coupling-derived groups

Same native column IDs across time/frequency and waveforms; panel-title colors
identify **Learned's own frozen groups**, not baseline groups. Solid/dashed curves
show real/imaginary coefficients. Frequency curves sample the finite-block complex
DTFT S_j(f)=ΣₙW[n,j]exp(-j2πfn)/√N with eightfold zero padding, no window or peak
normalization. At DFT knots S_j(k/N)=C[k,j], C=FW. Magnitude is |C|; power is |C|²;
Σₖ|C[k,j]|²=1. Dense DTFT samples are correlated interpolation, not new evidence.

The following permutation table explicitly maps plotted ranks to native basis IDs.
Groups were inferred from **nominal coupling**, independently of this sample channel;
Archived grouping is size-specific; new grouping is derived from train/discovery multipath
coupling by thresholded graph components, with diagonal excluded. Count can be one.
Freeze these groups across all views. They are approximate,
leaky and may occupy split/wrapped frequency regions, not a unique analytical construction.

A[i,j] couples transmitted basis j into received basis i: Hw_j=Σᵢw_i A[i,j].
The group view separates one |A|² sample, mean |A|² over multipath channels,
and Q=mean|WᴴT(τ,ν)W|² over unitary atoms. Titles give actual counts/laws. Q is not the normalized
multipath second moment. Neither statistic is |mean(complex A)|².
Replay retains nominal statistics. Fresh recomputes independent diagnostic test statistics
under the evaluation law, without changing W/support or selecting from test data.

The complex bridge Cᴴ(FHFᴴ)C=A is checked below; power spectra alone cannot reconstruct A.
The propagation view uses DISPLAY's first native column j and the same H as the channel
figure: received-index powers and coherently synthesized group spectra locate the leakage.
Group spectral **powers are not additive**: complex contributions have cross terms.


**Grouped energy/magnitude view (additional, not a replacement for complex spectra):**
one row per actual coupling group, matching the group colors/permutation/native-ID table.
Left: **every member**'s absolute N-point DFT coefficient energy |C[k,j]|², with
black arithmetic group mean. No individual peak normalization on this side; each
unit-norm column sums to1 over N DFT bins. Right: every member's finite-block
spectral **magnitude**, 20log₁₀(|S_j(f)|/max_f|S_j(f)|), individually peak-normalized
on the displayed32N-point grid and clipped at−60dB. Shapes here do not compare
absolute strength. Both axes use periodic f/Fs in[0,1); the endpoint1 labels the
wrap boundary, not an extra sample. Zero padding interpolates the finite-block
spectrum; it adds no physical frequency resolution. No window or retraining.
Rows/counts come from the frozen size-specific grouping, not the reference example.
The separate complex real/imaginary frequency figure above is retained.


**Individually identifiable examples:** EXAMPLE_COLUMNS in the single configuration
cell selects native Learned column IDs, independently of DISPLAY.basis_columns.
The selected examples are shown in frozen coupling order, at most two per group,
with an inline table linking **group, native j and permuted rank**. Defaults use
the lowest native ID of each frozen group; they are not selected for BER or appearance.
Black labeled lines in the grouped plot's right column identify those same examples
against all member curves. Each has its own row below: real/imaginary time basis,
readable spectral shape, and that exact transmitted column's |A[i,j]|² across the
received bases using the same displayed H. Titles repeat group/native j/rank;
received-basis bar colors match the frozen groups. Changing examples changes only
what is displayed; it does not change W, H, A, group inference, receiver or BER.
The separate actual complex-frequency comparison remains above.


**Readability variations for user review before final simplification:** PLOT_VARIANTS
chooses the enabled alternatives in this same configuration cell. OVERLAY8_COLUMNS
selects exactly8 native basis IDs; HALF_COLUMNS selects exactlyN/2 (32/64/128).
Every selected curve is overlaid together in each applicable panel, not split into
small subsets. Display follows the frozen coupling permutation; the full curve
key and inline index tables identify native j, group G and rank r. Color is a
stable native-ID color, not the group-color palette used in the group overview.
Line style is fixed by coupling rank. Default index choices are deterministic,
independent of appearance/BER. Dense overlays are retained for readability review;
use the labeled individual examples when curves overlap too heavily to trace.

The complex layouts separate real/imaginary time samples and actual complex DTFT
coefficients, without peak normalization. Energy/magnitude layouts instead show
absolute N-point |C|² and individually peak-normalized20log10 magnitude (−60dB
floor). Same32N-point interpolation grid; zero padding adds no physical resolution.
The coupling-group reference view remains an enabled alternative by default, and
all earlier coupling/individual/complex-basis figures remain available. These are
views of one Learned matrix per N, not comparisons among learned variants.


CFO shows distinct complex transmit W and receive Rᴴ columns. DD coupling-group,
unitary bridge and 8/N/2 DD overlays are inapplicable to rectangular W/R.

In [ ]:
workflow.show_basis_analysis()

## 4. BER versus SNR — mode, counts and stopping matter

Replay loads exact completed nominal counts; fresh always simulates new observations.
DD compares Learned/OTFS/AFDM at identical unit symbol energy, physical H/noise,
K and common stopped channel prefix per point. Fixed-support LMMSE and full-A
LMMSE remain separate; observations retain the complete physical channel.
CFO evaluates the frozen historical **direct R analysis**; no extra LMMSE,
classical/DD receiver equivalence or historical Eb/N0 curve reuse is invented.
New CFO and DD evaluations use **Es/N0**, raw QPSK/Gray16QAM slicing.

Fresh stops at common sample cap, error target for all arms, or batch-boundary time
cap. Minimum-coverage and target flags remain visible. Approximate channel-SE bands
are descriptive under adaptive stopping. Zero errors are unresolved: downward
triangles at 1/bits show resolution, not confidence limits; table gives conservative
independent-channel upper bounds. No universal win or training-seed uncertainty claim.

Replay's completed curves retain their original bootstrap/stopping flags. Modifying
training/channel/support settings never selects or relabels those saved curves.

In [ ]:
ber = workflow.evaluate_ber()
workflow.show_ber_figures()

## Complete

All figures above were generated inline from the selected basis through the same
DD plotting pipeline for load/train modes. Local ignored archives retain settings,
source hashes, counts, complex matrices, support/group mappings and training selection.
New default training automatically saves an inference checkpoint in its run archive;
set CHECKPOINT to that .npz with load_existing + fresh to evaluate it again.

CFO remains a separate verified W/R path with explicitly bounded support. Internal
interventions/structural-transfer studies stay available, outside advisor figures.

In [ ]:
display(JSON(workflow.finish(), expanded=True))
